# Exploratory Data Analysis

## Configuration

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.append("../utils")
from download_parquet import download_dataset
import numpy as np
import json
import re

    

DOWNLOAD_DATA = False
script_path = Path('..') / 'utils' / '00_donwload_parquet.py'
parquet_path = Path('..') / 'data' / 'clinical_trials.parquet'

c:\Users\orcor\anaconda3\envs\ds_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download Hugging Face Data (Optativo)

In [4]:

if DOWNLOAD_DATA:

    download_dataset()
else:
    print('No se descargaron los datos. Si deseas descargar los datos, establece DOWNLOAD_DATA en True.')

No se descargaron los datos. Si deseas descargar los datos, establece DOWNLOAD_DATA en True.


## Load Dataframe

In [2]:
df = pd.read_parquet(parquet_path)
df.head()

,nct_id,brief_title,official_title,overall_status,completion_date,brief_summary,conditions,keywords,phases,arm_groups,eligibility_criteria,min_age,std_ages,locations,condition_meshes,has_results
0,NCT00728442,Impact of OncoDoc2 on Guideline Compliance in ...,Impact of the OncoDoc2 Decision Support System...,COMPLETED,2010-04,The objective of the study is to evaluate how ...,"[""Breast Cancer""]","[""Clinical practice guidelines"", ""Guideline ad...","[""NA""]","[{""label"": ""1"", ""description"": ""medical decisi...","Inclusion Criteria:\n\n* Non metastatic, inclu...",NaN,"[""CHILD"", ""ADULT"", ""OLDER_ADULT""]","[{""status"": null, ""zip"": ""75020"", ""country"": ""...","[{""id"": ""D001943"", ""term"": ""Breast Neoplasms""}]",False
1,NCT01859442,The Effect of Cancer Therapies and Exercise on...,Phase 1 Study of Mitochondrial Energetics Afte...,COMPLETED,2013-07,The present study is a randomised control tria...,"[""Locally Advanced Rectal Cancer""]",[],"[""EARLY_PHASE1""]","[{""label"": ""Exercise group"", ""description"": ""6...",Inclusion Criteria:\n\n* diagnosis of T3N+ rec...,18 Years,"[""ADULT"", ""OLDER_ADULT""]","[{""status"": null, ""zip"": ""L97Al"", ""country"": ""...",[],False
2,NCT05741242,"A Phase 1b/2, Multi-center, Single Arm Study t...","A Phase 1b/2, Multi-center, Single Arm Study t...",ENROLLING_BY_INVITATION,2027-01,"A Phase 1b/2, Multi-center, Single Arm Study t...","[""Cancer"", ""Solid Tumor""]","[""personalized neoantigen vaccine""]","[""PHASE1"", ""PHASE2""]","[{""label"": ""Personalized Synthetic Long Peptid...",Patients must satisfy the following criteria t...,12 Years,"[""CHILD"", ""ADULT"", ""OLDER_ADULT""]","[{""status"": null, ""zip"": ""90212"", ""country"": ""...","[{""id"": ""D009369"", ""term"": ""Neoplasms""}]",False
3,NCT00838370,Pharmacogenomic and Pharmacokinetic Safety and...,Pharmacogenomic and Pharmacokinetic Safety and...,COMPLETED,2011-10,The primary purpose of this study is to prospe...,"[""Neoplasms""]","[""Pharmacogenetics"", ""cost-benefit analysis"", ...","[""PHASE1""]","[{""label"": ""DPYD*2A"", ""description"": ""Patients...",Inclusion Criteria:\n\n* Histological proof of...,18 Years,"[""ADULT"", ""OLDER_ADULT""]","[{""status"": null, ""zip"": ""1066CX"", ""country"": ...","[{""id"": ""D009369"", ""term"": ""Neoplasms""}]",False
4,NCT07538973,A Clinical Study to Evaluate the Pharmacokinet...,Phase I Clinical Study to Evaluate the Pharmac...,NOT_YET_RECRUITING,2027-03,Evaluation of the Pharmacokinetics and Safety ...,"[""Lung Cancer""]",[],"[""PHASE1""]","[{""label"": ""Anecatibin Fumarate Capsules (TQ-B...",Inclusion Criteria:\n\nAll subjects must meet ...,18 Years,"[""ADULT"", ""OLDER_ADULT""]","[{""status"": null, ""zip"": ""361000"", ""country"": ...","[{""id"": ""D008175"", ""term"": ""Lung Neoplasms""}]",False


## Análisis de Nulos

Para columnas escalares se detecta `NaN`/`None`. Para columnas que contienen arrays (listas), un valor "nulo" equivale a una lista vacía `[]`.

In [ ]:

# ── helpers ──────────────────────────────────────────────────────────────────

def _parse_if_json_list(x):
    """Intenta convertir x a lista Python. Retorna la lista (puede estar vacía) o None."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if isinstance(x, str):
        try:
            parsed = json.loads(x)
            if isinstance(parsed, list):
                return parsed
        except (json.JSONDecodeError, ValueError):
            pass
    return None  # escalar no-lista

def _is_array_col(series):
    """True si la columna contiene arrays (listas Python, ndarray o strings JSON de lista)."""
    for val in series.dropna().head(500):
        if _parse_if_json_list(val) is not None:   # [] vacío también es válido
            return True
    return False

def _is_empty_array(x):
    """True si el valor representa un array vacío ([], np.array([]), '[]', None excluido)."""
    parsed = _parse_if_json_list(x)
    return isinstance(parsed, list) and len(parsed) == 0

# ── clasificar columnas ──────────────────────────────────────────────────────

array_cols  = [col for col in df.columns if _is_array_col(df[col])]
scalar_cols = [col for col in df.columns if col not in array_cols]

print("Columnas array  :", array_cols)
print("Columnas escalar:", scalar_cols)

# ── nulos escalares (NaN / None) ─────────────────────────────────────────────

scalar_nulls     = df[scalar_cols].isnull().sum().rename("nulos")
scalar_nulls_pct = (df[scalar_cols].isnull().mean() * 100).rename("pct_nulos")

# ── "nulos" en arrays (lista vacía) ─────────────────────────────────────────

array_empty     = pd.Series(
    {col: df[col].apply(_is_empty_array).sum()        for col in array_cols},
    name="nulos"
)
array_empty_pct = pd.Series(
    {col: df[col].apply(_is_empty_array).mean() * 100 for col in array_cols},
    name="pct_nulos"
)

# ── reporte unificado ────────────────────────────────────────────────────────

null_report = pd.concat([
    pd.concat([scalar_nulls, scalar_nulls_pct], axis=1).assign(tipo_columna="escalar"),
    pd.concat([array_empty,  array_empty_pct],  axis=1).assign(tipo_columna="array"),
])

null_report["pct_nulos"] = null_report["pct_nulos"].round(2)
null_report = null_report.sort_values("pct_nulos", ascending=False)

print(f"\nTotal de registros: {len(df):,}\n")
null_report

Columnas array  : ['conditions', 'keywords', 'phases', 'arm_groups', 'std_ages', 'locations', 'condition_meshes']
Columnas escalar: ['nct_id', 'brief_title', 'official_title', 'overall_status', 'completion_date', 'brief_summary', 'eligibility_criteria', 'min_age', 'has_results']

Total de registros: 440,579



,nulos,pct_nulos,tipo_columna
keywords,157281,35.70,array
phases,101086,22.94,array
condition_meshes,93531,21.23,array
arm_groups,51101,11.60,array
locations,36630,8.31,array
min_age,28012,6.36,escalar
completion_date,14110,3.20,escalar
official_title,6852,1.56,escalar
eligibility_criteria,39,0.01,escalar
nct_id,0,0.00,escalar


In [6]:

# Mostrar eligibility_criteria de forma legible
for i, (idx, row) in enumerate(df[['nct_id', 'eligibility_criteria']].dropna().head(5).iterrows()):
    print(f"{'='*60}")
    print(f"NCT ID: {row['nct_id']}")
    print(f"{'='*60}")
    print(row['eligibility_criteria'])
    print()


NCT ID: NCT00728442
Inclusion Criteria:

* Non metastatic, including invasive and in situ, breast cancers as well as axillary cancer without breast tumor.
* At least one therapeutic MSM decision.

Exclusion Criteria:

* Breast disease without cancer
* Metastatic breast cancer
* Male breast cancer
* Breast cancer cases when medical records are not accessible to investigators
* Management but not therapeutic breast cancer decisions (diagnostic investigations, treatment follow-up, delayed decisions…)

NCT ID: NCT01859442
Inclusion Criteria:

* diagnosis of T3N+ rectal cancer
* above age of 18
* able to conduct a cardiopulmonary exercise test on a cycle ergometer

Exclusion Criteria:

* unable to conduct a cardiopulmonary exercise test on a cycle ergometer
* metastatic disease

NCT ID: NCT05741242
Patients must satisfy the following criteria to be enrolled in the protocol:

Main Inclusion Criterion:

Pancreatic

1. Patients who have local or metastatic pancreatic adenocarcinoma and have st

In [ ]:


ec = df['eligibility_criteria'].dropna()
total = len(ec)

# Debe contener 'Inclusion Criteria:' seguido de 'Exclusion Criteria:'
pattern = re.compile(r'Inclusion Criteria\s*:.*?Exclusion Criteria\s*:', re.DOTALL | re.IGNORECASE)

matches = ec.apply(lambda x: bool(pattern.search(x)))
no_match = (~matches).sum()

print(f"Total eligibility_criteria (sin nulos): {total:,}")
print(f"Con formato esperado (Inclusion + Exclusion): {matches.sum():,} ({matches.sum()/total*100:.2f}%)")
print(f"SIN formato esperado:                         {no_match:,} ({no_match/total*100:.2f}%)")

print("\n--- Ejemplos de NO match (primeros 5) ---")
for idx, val in ec[~matches].head(5).items():
    print(f"[{df.loc[idx, 'nct_id']}] {repr(val)}")
    print()


Total eligibility_criteria (sin nulos): 83,958
Con formato esperado (Inclusion + Exclusion): 72,626 (86.50%)
SIN formato esperado:                         11,332 (13.50%)

--- Ejemplos de NO match (primeros 5) ---
[NCT06806683] 'Inclusion Criteria\n\nParticipants must meet all of the following criteria:\n\n* Diagnosis: Confirmed head and neck cancer, including cancers of the oropharynx, larynx, hypopharynx, nasopharynx, salivary gland, unknown head and neck primary or oral cavity.\n* Age: 18 years or older.\n* Treatment: Undergoing or scheduled to undergo radiotherapy or chemoradiotherapy as part of their treatment\n\nExclusion Criteria\n\nParticipants will be excluded if they meet any of the following criteria:\n\n* Pregnancy: Pregnant or breastfeeding women.\n* Non-Compliance: Patients unwilling or unable to comply with study procedures and follow-ups.'

[NCT03979729] "Inclusion Criteria for Retrospective Component (Primary Objective):\n\n\\* Screening mammographic examinations perfo